In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

# =====================================================================
# Simple LSTM Binary Sentiment Classifier Demonstration
# =====================================================================

# 1. Dataset
# Simple movie review text reviews with labels (1 = Positive, 0 = Negative)
data = [
    ("I love this movie", 1),
    ("This film was great", 1),
    ("Superb acting and plot", 1),
    ("I hated this movie", 0),
    ("This film was terrible", 0),
    ("Bad acting and boring plot", 0)
]

# 2. Vocabulary Building
# Map every unique word in our corpus to a unique integer index
vocab = {"<PAD>": 0}
for sentence, _ in data:
    for word in sentence.lower().split():
        if word not in vocab:
            vocab[word] = len(vocab)

# Hyperparameters
VOCAB_SIZE = len(vocab)
EMBEDDING_DIM = 8
HIDDEN_DIM = 8
OUTPUT_DIM = 1
LEARNING_RATE = 0.05
EPOCHS = 100

print(f"Vocabulary Size: {VOCAB_SIZE} words")

# 3. Model Definition
class SimpleLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(SimpleLSTM, self).__init__()
        # Embedding converts token index to a continuous dense vector
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        # LSTM processes sequence vectors sequentially
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        # Fully connected layer predicts binary classification logit from final hidden state
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # Input shape: [batch_size, sequence_length]
        embedded = self.embedding(x)  # Shape: [batch_size, sequence_length, embedding_dim]

        # lstm_out: outputs of all time-steps
        # hidden: hidden state of final time-step
        lstm_out, (hidden, cell) = self.lstm(embedded)

        # Extract the last hidden state and pass to classifier
        last_hidden = hidden.squeeze(0)  # Shape: [batch_size, hidden_dim]
        out = self.fc(last_hidden)        # Shape: [batch_size, output_dim]
        return out

# Initialize model, loss, and optimizer
model = SimpleLSTM(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Helper function to convert sentence to token tensor
def sentence_to_tensor(sentence, vocab):
    indexes = [vocab[word] if word in vocab else 0 for word in sentence.lower().split()]
    return torch.tensor([indexes], dtype=torch.long)

# 4. Training Loop
print("\nTraining Simple LSTM Model...")
for epoch in range(EPOCHS):
    total_loss = 0
    for sentence, label in data:
        optimizer.zero_grad()

        # Convert text to tensor and target label to binary float tensor
        input_tensor = sentence_to_tensor(sentence, vocab)
        target_tensor = torch.tensor([[label]], dtype=torch.float)

        # Forward pass
        predictions = model(input_tensor)

        # Calculate loss & backpropagate
        loss = criterion(predictions, target_tensor)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:03d}/{EPOCHS} | Avg Loss: {total_loss/len(data):.4f}")

# 5. Testing Inference
print("\n--- Testing Inference ---")
test_sentences = [
    "I love this film",
    "This movie was bad"
]

model.eval()
with torch.no_grad():
    for test_sentence in test_sentences:
        input_tensor = sentence_to_tensor(test_sentence, vocab)
        output = model(input_tensor)

        # Convert logit output to probability [0.0, 1.0] using sigmoid
        probability = torch.sigmoid(output).item()
        sentiment = "Positive" if probability > 0.5 else "Negative"

        print(f"Sentence: '{test_sentence}' | Score: {probability:.4f} ({sentiment})")


Vocabulary Size: 16 words

Training Simple LSTM Model...
Epoch 020/100 | Avg Loss: 0.0022
Epoch 040/100 | Avg Loss: 0.0009
Epoch 060/100 | Avg Loss: 0.0005
Epoch 080/100 | Avg Loss: 0.0003
Epoch 100/100 | Avg Loss: 0.0002

--- Testing Inference ---
Sentence: 'I love this film' | Score: 0.9916 (Positive)
Sentence: 'This movie was bad' | Score: 0.0217 (Negative)
